In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.data import load_and_preprocess_data
from src.model import build_full_unet
from src.metrics import bce_dice_loss, dice_coefficient

print("Modules imported successfully!")

In [ ]:
X, Y = load_and_preprocess_data(data_dir="../data/kaggle_3m/", img_size=128)

print(f"Dataset ready in notebook!")
print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")

In [ ]:
model = build_full_unet(input_shape=(128, 128, 3))

model.compile(optimizer='adam', loss=bce_dice_loss, metrics=['accuracy', dice_coefficient])

model.summary()

In [ ]:
val_size = int(len(X) * 0.2)
X_val, Y_val = X[-val_size:], Y[-val_size:]

preds = model.predict(X_val)

tumor_indices = []
for i in range(len(Y_val)):
    if np.sum(Y_val[i]) > 0:  
        tumor_indices.append(i)

print(f"Found {len(tumor_indices)} validation images containing tumors out of {len(Y_val)} total.")

sample_indices = tumor_indices[:3]

fig, axes = plt.subplots(len(sample_indices), 3, figsize=(12, 4 * len(sample_indices)))

if len(sample_indices) == 1:
    axes = np.array([axes])

for row_idx, i in enumerate(sample_indices):
    # Original MRI Image
    axes[row_idx, 0].imshow(X_val[i])
    axes[row_idx, 0].set_title(f"Original MRI (Index {i})")
    axes[row_idx, 0].axis('off')
    
    # Ground Truth Mask 
    axes[row_idx, 1].imshow(Y_val[i].squeeze(), cmap='gray')
    axes[row_idx, 1].set_title("True Mask (Doctor's Note)")
    axes[row_idx, 1].axis('off')
    
    # Model's Prediction
    pred_mask = (preds[i].squeeze() > 0.5).astype(np.float32)
    axes[row_idx, 2].imshow(pred_mask, cmap='gray')
    axes[row_idx, 2].set_title("Model Prediction")
    axes[row_idx, 2].axis('off')

plt.tight_layout()
plt.show()